# Solar Filament V10 — CPU Mask-Quality Learning & V7/V8 Consensus

**Objective:** Improve an actual, provenance-linked **0.34-level V7 pipeline** without a GPU. V9 returned 0.31 because it selected the older V4 baseline; its independent LightGBM proposals dramatically increased false positives. V10 does not train another pixel-only model: it learns **whether an entire proposed filament mask is likely to match a human-annotated instance**, combining full-disk YOLO, native-tile YOLO and (optionally) the 13-epoch V8 specialist.

**Method:** mask-quality prediction inspired by [Mask Scoring R-CNN (CVPR 2019)](https://openaccess.thecvf.com/content_CVPR_2019/html/Huang_Mask_Scoring_R-CNN_CVPR_2019_paper.html), adapted to CPU-only histogram boosting and inter-model mask agreement. The mask-quality model is fitted on a reserved subset of the **official training annotations**, never test labels. Configuration selection uses another distinct subset of the original 97 tune observations. Previously inspected audit images are a diagnostic, not a fresh independent holdout.

**No guarantee:** The observed 0.34 is the best user-confirmed score, not a score of V10. The notebook never accesses the unverified 0.55 precomputed submission. Nothing is promoted unless it beats the reconstructed V7 or V4 *in local validation* with a positive margin; hidden public PQ may differ.

**Modes:** `ALL` (prepare, fit quality scorer, select, export), `FIT` (fit and select, no test inference), `SUBMIT` (reuse selection/model from a saved V10 output). Every inference is from official images and user-produced model weights; expensive proposals and features are cached for CPU-only resume.

In [ ]:
import os, sys, json, time, hashlib, random, shutil, subprocess, importlib.util, gc
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from IPython.display import display

# ---------------- USER CONFIG ----------------
MODE = 'ALL'                      # ALL | FIT | SUBMIT
DATA_ROOT = Path('/kaggle/input/datasets/lighittha/solar3k/MAGFiLO_1.0_Kaggle_2026')
V4_ROOT = Path('/kaggle/input/datasets/lighittha/solaroutsp2/solar_cpu_v4')
V7_ROOT = None                    # Auto-search: folder with v7_config.json (and ideally selection.json)
V8_WEIGHTS = None                 # Auto-search: .../solar_cpu_v8/runs/tile_specialist/weights/best.pt
PREVIOUS_V10_ROOT = None          # Optional attached prior V10 folder
OUT = Path('/kaggle/working/solar_cpu_v10' if Path('/kaggle/working').exists() else '/mnt/data/solar_cpu_v10')
ENABLE_V8 = True                 # Works if legitimate V8 specialist weights are attached; otherwise V7-only
SEED = 2026
MIN_GAIN_TO_PROMOTE = 0.010      # Absolute PQ gain on 42 held-out tune observations
RAW_CONF, RAW_IOU, GLOBAL_IMGSZ, MAX_DET = .10, .82, 1024, 65
QUALITY_FIT_FRACTION = .57      # 55 tune observations for quality fit / 42 for recipe selection
MAX_GLOBAL=65; MAX_V7_TILES=85; MAX_V8_TILES=90
TILE_SIDE=1280; TILE_IMGSZ=1024
AUTO_INSTALL=True
# ---------------------------------------------
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); cv2.setNumThreads(0)
for name,pkg in [('pycocotools','pycocotools'),('ultralytics','ultralytics>=8.3,<9')]:
    if importlib.util.find_spec(name) is None:
        if not AUTO_INSTALL:raise ImportError(pkg)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import torch, ultralytics
from ultralytics import YOLO
from pycocotools import mask as coco
if hasattr(torch,'set_num_threads'):torch.set_num_threads(min(4,os.cpu_count() or 2))
H=W=2048
print('V10 CPU',torch.__version__,ultralytics.__version__,'mode',MODE,'output',OUT)

## 1 — Discover attached assets and enforce the original grouped split

Any old `solaroutsp2`, `solarouts`, or individually versioned Kaggle dataset is supported if the required files are present. Missing V7 selection is never silently presented as the 0.34 recipe: it is **reconstructed on the quality-fit portion of the tune split**. V8 is optional but recommended.

In [ ]:
def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()

def find_root(requested,marker, extra=None):
    if requested is not None:
        requested=Path(requested)
        if (requested/marker).exists() and (extra is None or (requested/extra).exists()):return requested
    options=[]
    for base in [Path('/kaggle/input'),Path('/mnt/data')]:
        if base.exists():
            options += [p.parents[len(Path(marker).parts)-1] for p in base.rglob(marker) if extra is None or (p.parents[len(Path(marker).parts)-1]/extra).exists()]
    if not options:raise FileNotFoundError(f'Missing {marker}; attach source dataset')
    if len(options)>1:print('Multiple compatible',marker,'detected; choosing first:',[str(p) for p in options[:6]])
    return sorted(options,key=lambda p:(-int('solar_cpu_v4' in str(p)),str(p)))[0]

DATA=find_root(DATA_ROOT,'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json','test/test_images')
V4=find_root(V4_ROOT,'split_manifest.json','runs/yolo11n-seg/weights/best.pt')
OLD_V7=None
if V7_ROOT is not None and (Path(V7_ROOT)/'v7_config.json').exists():OLD_V7=Path(V7_ROOT)
else:
    roots=[]
    for base in [Path('/kaggle/input'),Path('/mnt/data')]:
        if base.exists(): roots += [p.parent for p in base.rglob('v7_config.json')]
    if roots:
        roots.sort(key=lambda p:(not (p/'selection.json').is_file(), 'solar_cpu_v7' not in str(p),str(p)))
        OLD_V7=roots[0]

V8_PATH=Path(V8_WEIGHTS) if V8_WEIGHTS else None
if ENABLE_V8 and (V8_PATH is None or not V8_PATH.is_file()):
    options=[]
    for base in [Path('/kaggle/input'),Path('/mnt/data')]:
        if base.exists():options += [p for p in base.rglob('tile_specialist/weights/best.pt')]
    V8_PATH=sorted(options,key=str)[0] if options else None
if V8_PATH is None:ENABLE_V8=False

RAW=json.loads((DATA/'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
SPLIT=json.loads((V4/'split_manifest.json').read_text())
TRAIN={p.stem:p for p in (DATA/'train/train_images').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}
TEST={p.stem:p for p in (DATA/'test/test_images').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}
assert len(TEST)==180 and len(TRAIN)==707 and set().union(*(set(SPLIT[k]) for k in ('train','tune','audit')))==set(TRAIN)
assert not (set(SPLIT['train'])&set(SPLIT['tune']) or set(SPLIT['audit'])&set(SPLIT['tune']) or set(SPLIT['audit'])&set(SPLIT['train']))
for a in SPLIT:
    for b in SPLIT:
        if a!=b:assert not ({x[:8] for x in SPLIT[a]}&{x[:8] for x in SPLIT[b]})
AB=defaultdict(list)
for a in RAW['annotations']: AB[str(a['image_id'])].append(a)
OBS=defaultdict(list)
for im in RAW['images']:
    stem=Path(im['file_name']).stem
    if stem in TRAIN:OBS[stem].append((str(im['id']),AB[str(im['id'])]))
V4_CKPT=V4/'runs/yolo11n-seg/weights/best.pt'
V4_SHA=digest(V4_CKPT); SPLIT_SHA=digest(V4/'split_manifest.json')
V8_SHA=digest(V8_PATH) if ENABLE_V8 else None
V7CFG=json.loads((OLD_V7/'v7_config.json').read_text()) if OLD_V7 else {}
if V7CFG and (V7CFG.get('checkpoint_sha256')!=V4_SHA or V7CFG.get('split_sha256')!=SPLIT_SHA):
    print('V7 folder incompatible with actual checkpoint/split; ignoring:',OLD_V7)
    OLD_V7=None;V7CFG={}
TILE_SIDE=int(V7CFG.get('tile_side',TILE_SIDE));TILE_IMGSZ=int(V7CFG.get('tile_imgsz',TILE_IMGSZ))
TILE_SCHEME=V7CFG.get('tile_scheme','FIVE')
assert TILE_SCHEME in ('FOUR','FIVE')
V7_SELECTION=None
if OLD_V7 and (OLD_V7/'selection.json').is_file():
    d=json.loads((OLD_V7/'selection.json').read_text())
    if d.get('source_model_sha256')==V4_SHA and d.get('split_sha256')==SPLIT_SHA:
        V7_SELECTION=d.get('strategy')
print('V4',V4,'V7',OLD_V7,'saved V7 recipe:',V7_SELECTION is not None)
print('V8',V8_PATH if ENABLE_V8 else 'NOT ATTACHED (V7-only branch)')
print('Splits:',{k:len(v) for k,v in SPLIT.items()})

## 2 — Mask and matching utilities: COCO compressed RLE, disjoint instances, per-annotator PQ

The PQ proxy is explicitly labeled as such; do not confuse it with the hidden Kaggle public score. We use one-to-one IoU > 0.5 assignment and verify all masks are nonoverlapping. No test label file is loaded.

In [ ]:
def to_rle(v):
    if isinstance(v,dict):return {'size':[H,W],'counts':v['counts'].encode('ascii') if isinstance(v['counts'],str) else v['counts']}
    return {'size':[H,W],'counts':v.encode('ascii') if isinstance(v,str) else v}
def to_ascii(r):return r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else str(r['counts'])
def encode(m):return coco.encode(np.asfortranarray(np.asarray(m,dtype=np.uint8)))
def disjoint(rles):
    if len(rles)<2:return True
    iou=np.asarray(coco.iou(rles,rles,[0]*len(rles)))
    np.fill_diagonal(iou,0.)
    return bool((iou<=0).all())
def ann_rle(a):
    seg=a.get('segmentation')
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):return coco.frPyObjects(seg,H,W)
        return to_rle(seg)
    if not seg:return None
    if isinstance(seg[0],(float,int)):seg=[seg]
    polys=[]
    for q in seg:
        if len(q)<6 or len(q)%2:continue
        arr=np.asarray(q,dtype=np.float64).reshape(-1,2)
        if np.isfinite(arr).all():polys.append(np.clip(arr,[0,0],[W-1,H-1]).ravel().tolist())
    if not polys:return None
    r=coco.frPyObjects(polys,H,W)
    return coco.merge(r) if isinstance(r,list) else r
GT={}
def gt_for(stem):
    if stem not in GT:
        GT[stem]=[(uid,[r for a in anns if (r:=ann_rle(a)) is not None]) for uid,anns in OBS[stem]]
    return GT[stem]
def count_matching(gt,pred):
    if not gt and not pred:return np.zeros(4,float)
    if not gt:return np.array([0,0,len(pred),0.],float)
    if not pred:return np.array([0,0,0,len(gt)],float)
    iou=np.asarray(coco.iou(gt,pred,[0]*len(pred)),float)
    rows,cols=linear_sum_assignment(-iou)
    good=iou[rows,cols]>0.5
    tp=int(good.sum())
    return np.array([iou[rows[good],cols[good]].sum(),tp,len(pred)-tp,len(gt)-tp],float)
def summary(total):
    si,tp,fp,fn=map(float,total);den=tp+.5*(fp+fn)
    return {'pq':float(si/den) if den else 0.,'sq':float(si/tp) if tp else 0.,
            'rq':float(tp/den) if den else 0.,'tp':int(tp),'fp':int(fp),'fn':int(fn)}
def score_strategy(stems, fn):
    counts=np.zeros(4,float)
    for stem in stems:
        masks=fn(stem)
        assert disjoint(masks),('overlap',stem)
        for _,truth in gt_for(stem):counts+=count_matching(truth,masks)
    return summary(counts)
# Micro-tests
m=np.zeros((H,W),np.uint8);m[17:55,12:85]=1; r=encode(m)
assert abs(summary(count_matching([r],[r]))['pq']-1)<1e-10
assert count_matching([r],[])[3]==1
assert disjoint([r])
print('RLE and instance-PQ unit tests passed')
del m

## 3 — Provenance-linked CPU model proposals: global, V7 tiles, V8 specialist tiles

Reuse existing saved V4/V7 caches, if valid. Otherwise compute missing predictions from the attached weights. V8 uses the same CLAHE-mix preprocessing it was trained with; V7 uses unprocessed H-alpha pixels. Cache provenance includes checkpoint SHA, inference size, windows and preprocessing.

In [ ]:
def windows():
    v=[(0,0),(W-TILE_SIDE,0),(0,H-TILE_SIDE),(W-TILE_SIDE,H-TILE_SIDE)]
    if TILE_SCHEME=='FIVE':v.append(((W-TILE_SIDE)//2,(H-TILE_SIDE)//2))
    return list(dict.fromkeys(v))
WINDOWS=windows()
HASHES={'global':V4_SHA,'v7':V4_SHA,'v8':V8_SHA}
RUN_CFG={'version':'v10-mask-quality-2','v4sha':V4_SHA,'v8sha':V8_SHA,'splitsha':SPLIT_SHA,
         'tile_side':TILE_SIDE,'tile_imgsz':TILE_IMGSZ,'scheme':TILE_SCHEME,'raw_conf':RAW_CONF,'raw_iou':RAW_IOU}
(OUT/'contract.json').write_text(json.dumps(RUN_CFG,indent=2))
PREV10=None
if PREVIOUS_V10_ROOT:
    PREV10=Path(PREVIOUS_V10_ROOT)
    assert json.loads((PREV10/'contract.json').read_text())==RUN_CFG,'Previous V10 belongs to different assets'
else:
    for base in [Path('/kaggle/input')]:
        if base.exists():
            for p in sorted(base.rglob('contract.json')):
                try:
                    if json.loads(p.read_text())==RUN_CFG:PREV10=p.parent;break
                except Exception:pass
P_DIR=OUT/'proposals'; P_DIR.mkdir(exist_ok=True)
NET={}
def net(kind):
    if kind not in NET:NET[kind]=YOLO(str(V8_PATH if kind=='v8' else V4_CKPT))
    return NET[kind]
def preprocess_tile(gray):
    equal=cv2.createCLAHE(clipLimit=1.5,tileGridSize=(8,8)).apply(gray)
    return cv2.addWeighted(gray,.65,equal,.35,0)
def meta(kind,stem):
    return {'version':RUN_CFG['version'],'kind':kind,'stem':stem,'model_sha':HASHES[kind],
            'imgsz':GLOBAL_IMGSZ if kind=='global' else TILE_IMGSZ,
            'windows':[] if kind=='global' else [list(w) for w in WINDOWS],
            'preprocess':'clahe_mix' if kind=='v8' else 'none'}
def fcache(kind,stem):return P_DIR/(kind+'__'+stem+'.json')
def normalize_candidates(source,kind,cap=True):
    out=[]
    for p in source:
        if not {'counts','score'}.issubset(p):continue
        score=float(p['score']);area=int(p.get('area',coco.area(to_rle(p['counts']))))
        if not np.isfinite(score) or score<RAW_CONF or area<120:continue
        out.append({'counts':p['counts'],'score':score,'area':area,'source':kind,
                    'edge':bool(p.get('edge',p.get('truncated',False)))})
    out.sort(key=lambda p:(-p['score'],-p['area']))
    return out[:{'global':MAX_GLOBAL,'v7':MAX_V7_TILES,'v8':MAX_V8_TILES}[kind]] if cap else out
def old_v4_cache(stem):
    p=V4/'cache/yolo11n-seg'/(stem+'.json')
    if not p.exists():return None
    try:
        d=json.loads(p.read_text())
        if (d.get('stem')==stem and d.get('checkpoint_sha256')==V4_SHA and d.get('predict_imgsz')==1024
            and d.get('tta') is False and d.get('source')=='LIVE_IMAGE_INFERENCE'):
            return normalize_candidates(d['proposals'],'global')
    except Exception:pass
    return None

def old_v7_cache(stem):
    if not OLD_V7:return None
    p=OLD_V7/'tiles'/(stem+'.json')
    if not p.exists():return None
    try:
        d=json.loads(p.read_text())
        if (d.get('stem')==stem and d.get('kind')=='tile'
            and d.get('checkpoint_sha256')==V4_SHA and d.get('source')=='LIVE_TEST_OR_TRAIN_JPEG_INFERENCE'):
            # Only reuse if geometry/preprocessing and hash match V7's saved contract.
            if int(d.get('tile_side',-1))==TILE_SIDE and int(d.get('tile_imgsz',-1))==TILE_IMGSZ:
                return normalize_candidates(d['proposals'],'v7',cap=False)
    except Exception:pass
    return None

@torch.inference_mode()
def infer_frame(gray,kind,x=0,y=0):
    source=preprocess_tile(gray) if kind=='v8' else gray
    rgb=cv2.cvtColor(source,cv2.COLOR_GRAY2BGR)
    size=GLOBAL_IMGSZ if kind=='global' else TILE_IMGSZ
    result=net(kind).predict(source=rgb,imgsz=size,conf=RAW_CONF,iou=RAW_IOU,
                             retina_masks=True,agnostic_nms=True,max_det=MAX_DET,
                             device='cpu',verbose=False)[0]
    if result.masks is None or result.boxes is None:return []
    masks=result.masks.data.detach().cpu().numpy()
    confs=result.boxes.conf.detach().cpu().numpy()
    assert len(masks)==len(confs)
    props=[]
    for mask,prob in zip(masks,confs):
        mask=np.uint8(mask>.5)
        if mask.shape!=gray.shape:
            mask=cv2.resize(mask,(gray.shape[1],gray.shape[0]),interpolation=cv2.INTER_NEAREST)
        if not mask.any():continue
        if kind=='global':full=mask;edge=False
        else:
            edge=bool((x>0 and mask[:,:12].any()) or (y>0 and mask[:12,:].any()) or
                     (x+TILE_SIDE<W and mask[:,-12:].any()) or (y+TILE_SIDE<H and mask[-12:,:].any()))
            full=np.zeros((H,W),np.uint8);full[y:y+TILE_SIDE,x:x+TILE_SIDE]=mask
        rr=encode(full)
        props.append({'counts':to_ascii(rr),'score':float(prob),'area':int(coco.area(rr)),'source':kind,'edge':edge})
    return normalize_candidates(props,kind,cap=False)

PROPS_MEM={}
def get_proposals(stem,images,kind):
    key=(kind,stem)
    if key in PROPS_MEM:return PROPS_MEM[key]
    assert kind!='v8' or ENABLE_V8
    for path in [fcache(kind,stem),(PREV10/'proposals'/(kind+'__'+stem+'.json')) if PREV10 else None]:
        if path is None or not path.is_file():continue
        try:
            d=json.loads(path.read_text())
            if d['meta']==meta(kind,stem):
                PROPS_MEM[key]=d['proposals'];return PROPS_MEM[key]
        except Exception:pass
    preds=old_v4_cache(stem) if kind=='global' else old_v7_cache(stem) if kind=='v7' else None
    if preds is None:
        gray=cv2.imread(str(images[stem]),cv2.IMREAD_GRAYSCALE)
        if gray is None or gray.shape!=(H,W):raise ValueError(f'Expected 2048^2 grayscale: {stem}')
        if kind=='global':preds=infer_frame(gray,kind)
        else:
            preds=[]
            for x,y in WINDOWS:preds.extend(infer_frame(gray[y:y+TILE_SIDE,x:x+TILE_SIDE],kind,x,y))
    preds=normalize_candidates(preds,kind,cap=False)
    fcache(kind,stem).write_text(json.dumps({'meta':meta(kind,stem),'proposals':preds},separators=(',',':')))
    PROPS_MEM[key]=preds
    return preds

def get_all(stem,images):
    out=get_proposals(stem,images,'global')+get_proposals(stem,images,'v7')[:MAX_V7_TILES]
    if ENABLE_V8:out+=get_proposals(stem,images,'v8')[:MAX_V8_TILES]
    return out
print('Cache/config validation initialized; windows',WINDOWS,'V8',ENABLE_V8)

## 4 — Exact V4 control, saved V7 control, and V7 reconstruction if needed

V7 may have an old `selection.json` on one dataset version but not another. If none is found, reconstruct its selection on the **quality-model fitting subset only** and report this as `v7_reconstructed` — never mislabel it as the previously submitted 0.34 CSV. The original V4 control is also always evaluated.

In [ ]:
def old_v4(stem,images):
    ranked=sorted([p for p in get_proposals(stem,images,'global') if p['score']>=.32 and p['area']>=350],
                  key=lambda p:(-p['score'],-p['area']))
    occ=np.zeros((H,W),np.uint8);used=[];out=[]
    for p in ranked:
        rr=to_rle(p['counts'])
        if used and float(np.max(coco.iou([rr],used,[0]*len(used))))>=.5:continue
        used.append(rr)
        mask=coco.decode(rr).astype(bool)&(occ==0)
        if mask.sum()<350:continue
        occ[mask]=1;out.append(encode(mask))
    return out

def keep_disjoint(candidates, min_area=350, nms_iou=.50,containment=.88,min_free=.50):
    occ=np.zeros((H,W),np.uint8);used=[];out=[]
    for p in sorted(candidates,key=lambda q:(-q['rank'],-q['area'])):
        if p['area']<min_area:continue
        rr=to_rle(p['counts'])
        if used:
            iou=np.asarray(coco.iou([rr],used,[0]*len(used)))[0]
            size=np.asarray(coco.area(used),float)
            inter=iou*(size+p['area'])/(1.+iou+1e-12)
            if np.any(iou>=nms_iou) or np.any(inter/max(p['area'],1)>=containment):continue
        mm=coco.decode(rr).astype(bool)
        free=mm&(occ==0)
        n=int(free.sum())
        if n<min_area or n/max(1,p['area'])<min_free:continue
        occ[free]=1;used.append(rr);out.append(encode(free))
    assert disjoint(out)
    return out

def v7_recipe(stem,images,cfg):
    if cfg=='v4':return old_v4(stem,images)
    items=[]
    for p in get_proposals(stem,images,'global')+(get_proposals(stem,images,'v7') if cfg['mode']!='global' else []):
        tile=p['source']=='v7'
        if p['score']<(cfg['tile_conf'] if tile else cfg['global_conf']):continue
        if p['area']<cfg['min_area'] or (tile and cfg['seam_guard'] and p['edge']):continue
        items.append({**p,'rank':p['score']+(cfg['tile_bonus'] if tile else 0.)})
    return keep_disjoint(items,min_area=cfg['min_area'],nms_iou=cfg['nms_iou'],
                         containment=cfg['containment'],min_free=cfg['min_free_fraction'])

TUNESTEMS=sorted(SPLIT['tune']);random.Random(SEED+915).shuffle(TUNESTEMS)
nfit=int(round(len(TUNESTEMS)*QUALITY_FIT_FRACTION))
FIT_STEMS=sorted(TUNESTEMS[:nfit]);SELECT_STEMS=sorted(TUNESTEMS[nfit:])
assert not (set(FIT_STEMS)&set(SELECT_STEMS))
assert len(FIT_STEMS)>=30 and len(SELECT_STEMS)>=20
V7_GRID=[{'mode':'tiles','global_conf':g,'tile_conf':t,'tile_bonus':b,
          'min_area':350,'seam_guard':guard,'nms_iou':nms,'containment':.88,'min_free_fraction':.50}
          for g in (.28,.36) for t in (.35,.48,.62) for b in (0.,.10)
          for guard in (False,True) for nms in (.50,.70)]

CONTROL_FILE=OUT/'v7_control.json'
if MODE=='SUBMIT':
    if CONTROL_FILE.exists():V7_CONTROL=json.loads(CONTROL_FILE.read_text())
    elif PREV10 and (PREV10/'v7_control.json').exists():V7_CONTROL=json.loads((PREV10/'v7_control.json').read_text())
    else:raise FileNotFoundError('Missing frozen v7_control.json; run FIT/ALL first')
else:
    if V7_SELECTION is not None:
        ctrl=V7_SELECTION;label='saved_v7_exact'
        print('Using hash-verified saved V7 selection')
    else:
        print('Reconstructing V7 from verified V4/V7 inference; this is NOT the exact prior 0.34 CSV')
        for s in tqdm(FIT_STEMS,desc='V7 CPU proposals for baseline reconstruction'):
            get_proposals(s,TRAIN,'global');get_proposals(s,TRAIN,'v7')
        baseline_score=score_strategy(FIT_STEMS,lambda s:old_v4(s,TRAIN))['pq']
        choice='v4';best=baseline_score
        for p in tqdm(V7_GRID,desc='V7 train-only baseline reconstruction'):
            z=score_strategy(FIT_STEMS,lambda s:v7_recipe(s,TRAIN,p))['pq']
            if z>best:best=z;choice=p
        ctrl=choice;label='v7_reconstructed_fit_only' if isinstance(choice,dict) else 'v4'
        print('Reconstruction fit-PQ',best,'V4 fit-PQ',baseline_score)
    V7_CONTROL={'strategy':ctrl,'label':label,'v4_sha':V4_SHA,'split_sha':SPLIT_SHA}
    CONTROL_FILE.write_text(json.dumps(V7_CONTROL,indent=2))
assert V7_CONTROL['v4_sha']==V4_SHA and V7_CONTROL['split_sha']==SPLIT_SHA

def control_masks(stem,images):return v7_recipe(stem,images,V7_CONTROL['strategy'])
print('V7 control identity:',V7_CONTROL['label'],'quality fit images',len(FIT_STEMS),'selection images',len(SELECT_STEMS))

## 5 — A new **instance-quality learner**, not another foreground pixel classifier

Features include detection confidence, mask size and shape, H-alpha contrast relative to nearby pixels, radial position, and **support from independent model predictions**. The supervised label is whether a proposal matches an actual instance at IoU > 0.5 across annotated versions of that physical image. Trained only on the first portion of the original tune split; candidate selection uses the remaining tune observations.

In [ ]:
FEATURE_NAMES=['confidence','log_area','log_aspect','bbox_fill','edge','center_distance','mean_inside','mean_ring',
               'contrast','darkness','radial','source_global','source_v7','source_v8','support_other_iou',
               'support_other_score','support_same_iou','similar_count','width','height']
FEAT_DIR=OUT/'candidate_features';FEAT_DIR.mkdir(exist_ok=True)
FEAT_VER='v10-maskquality-features-2'
FEAT_MEM={}
def feature_file(stem):return FEAT_DIR/(stem+'.json')
def feature_key(stem):return {'config':RUN_CFG,'stem':stem,'version':FEAT_VER}

def build_features(stem,images):
    key=feature_key(stem)
    for file in [feature_file(stem),(PREV10/'candidate_features'/(stem+'.json')) if PREV10 else None]:
        if file is None or not file.is_file():continue
        try:
            d=json.loads(file.read_text())
            if d['meta']==key:return d['rows']
        except Exception:pass
    gray=cv2.imread(str(images[stem]),cv2.IMREAD_GRAYSCALE)
    if gray is None or gray.shape!=(H,W):raise RuntimeError('Unexpected image shape '+stem)
    enhanced=cv2.createCLAHE(clipLimit=1.5,tileGridSize=(8,8)).apply(gray)
    background=cv2.GaussianBlur(enhanced,(0,0),sigmaX=8)
    candidates=get_all(stem,images)
    rles=[to_rle(p['counts']) for p in candidates]
    n=len(candidates);box=np.asarray(coco.toBbox(rles),float) if n else np.zeros((0,4))
    areas=np.asarray([p['area'] for p in candidates],float)
    support=np.asarray(coco.iou(rles,rles,[0]*n)) if n else np.zeros((0,0))
    rows=[]
    for i,p in enumerate(candidates):
        rr=rles[i];area=max(1.,areas[i]);x,y,bw,bh=box[i];x0,y0=int(x),int(y)
        x1=min(W,int(np.ceil(x+bw)));y1=min(H,int(np.ceil(y+bh)))
        # Crop to the mask bounding rectangle. Decode is exact at native resolution.
        mask=coco.decode(rr).astype(bool)
        x0=max(0,x0);y0=max(0,y0)
        mm=mask[y0:y1,x0:x1];patch=gray[y0:y1,x0:x1];bg=background[y0:y1,x0:x1]
        if not mm.any():continue
        inside=float(np.mean(patch[mm]))
        ring_patch=cv2.dilate(np.uint8(mm),np.ones((7,7),np.uint8))>0
        ring=ring_patch&~mm
        outside=float(np.mean(patch[ring])) if ring.sum()>5 else inside
        darkness=float(np.mean(bg[mm].astype(float)-enhanced[y0:y1,x0:x1][mm].astype(float)))
        other=np.asarray([j!=i and candidates[j]['source']!=p['source'] for j in range(n)],bool)
        same=np.asarray([j!=i and candidates[j]['source']==p['source'] for j in range(n)],bool)
        ious=support[i]
        oth_iou=float(ious[other].max()) if other.any() else 0.
        oth_score=float(max([candidates[j]['score'] for j in np.flatnonzero(other) if ious[j]>.2],default=0.))
        same_iou=float(ious[same].max()) if same.any() else 0.
        center=np.array([(x+bw/2)/W,(y+bh/2)/H]);radius=float(np.linalg.norm(center-[.5,.5])*2**.5)
        ff=[float(p['score']),float(np.log1p(area)),float(np.log((max(bw,bh)+1)/(min(bw,bh)+1))),
            float(area/max(1.,bw*bh)),float(p['edge']),float(np.linalg.norm(center-[.5,.5])),
            inside/255.,outside/255.,(outside-inside)/64.,darkness/64.,radius,
            float(p['source']=='global'),float(p['source']=='v7'),float(p['source']=='v8'),
            oth_iou,oth_score,same_iou,float((ious>.3).sum()-1),float(bw/W),float(bh/H)]
        assert len(ff)==len(FEATURE_NAMES) and np.isfinite(ff).all()
        rows.append({'counts':p['counts'],'score':p['score'],'area':p['area'],'source':p['source'],
                     'edge':p['edge'],'features':ff})
    feature_file(stem).write_text(json.dumps({'meta':key,'rows':rows},separators=(',',':')))
    return rows

def features(stem,images):
    if (stem,'f') not in FEAT_MEM:FEAT_MEM[(stem,'f')]=build_features(stem,images)
    return FEAT_MEM[(stem,'f')]

def make_labels(stem,rows):
    if not rows:return np.zeros(0,dtype=np.int32)
    rr=[to_rle(x['counts']) for x in rows]
    agree=np.zeros(len(rr),float)
    for _,gt in gt_for(stem):
        if not gt:continue
        overlaps=np.asarray(coco.iou(gt,rr,[0]*len(rr)),float)
        agree+=(overlaps.max(axis=0)>0.5)
    agree/=max(1,len(gt_for(stem)))
    # Require agreement with at least half of the available annotator versions.
    return np.uint8(agree>=0.5)

QMODEL=OUT/'mask_quality_model.pkl';QMODEL_META=OUT/'mask_quality_metadata.json'
if MODE in ('ALL','FIT'):
    xx=[];yy=[]
    for stem in tqdm(FIT_STEMS,desc='Generate supervised instance-quality examples'):
        rows=features(stem,TRAIN)
        if rows:
            xx.extend(r['features'] for r in rows)
            yy.extend(map(int,make_labels(stem,rows)))
    X=np.asarray(xx,np.float32);Y=np.asarray(yy,np.int32)
    print('Quality-training candidate masks',len(Y),'positive fraction',round(float(Y.mean()),4))
    assert len(Y)>40 and len(set(Y))==2,'Need at least one positive and negative quality label'
    model=HistGradientBoostingClassifier(max_iter=140,max_leaf_nodes=15,max_depth=5,l2_regularization=5.,
                                         learning_rate=.05,min_samples_leaf=20,random_state=SEED)
    sample_weights=np.where(Y==1,np.clip((Y==0).sum()/max(1,(Y==1).sum()),1.,8.),1.)
    model.fit(X,Y,sample_weight=sample_weights)
    import joblib
    joblib.dump(model,QMODEL)
    QMODEL_META.write_text(json.dumps({'version':FEAT_VER,'features':FEATURE_NAMES,'contract':RUN_CFG,
                            'fit_images':FIT_STEMS,'count':len(Y),'positive_fraction':float(Y.mean()),
                            'model_sha':digest(QMODEL)},indent=2))
else:
    import joblib
    origin=QMODEL if QMODEL.is_file() else (PREV10/'mask_quality_model.pkl') if PREV10 else None
    if not origin or not origin.exists():raise FileNotFoundError('Attach previous V10 fitted scorer; run MODE=FIT first')
    model=joblib.load(origin)
    mpath=QMODEL_META if QMODEL_META.exists() else PREV10/'mask_quality_metadata.json'
    m=json.loads(mpath.read_text())
    assert m['version']==FEAT_VER and m['contract']==RUN_CFG and m['features']==FEATURE_NAMES
print('Learned full-instance quality model ready')

## 6 — Conservative independent-mask selector and held-out recipe comparison

The learned score is **not** a direct claim of true IoU; use it as an empirical selection feature. Baselines: exact V4 and recovered/reconstructed V7. Experiments: global+V7 masks scored by quality, then optionally V8 specialist masks, with standard disjoint RLE and NMS. A new candidate is promoted only if it exceeds the strongest available baseline by `MIN_GAIN_TO_PROMOTE` on 42 tune-only observations. The 97-image historical audit is never used for choosing parameters.

In [ ]:
QUALITY_PROBS_CACHE={}
def rescore(stem,images,params):
    rows=features(stem,images)
    if not rows:return []
    if stem not in QUALITY_PROBS_CACHE:
        X=np.asarray([x['features'] for x in rows],np.float32)
        QUALITY_PROBS_CACHE[stem]=model.predict_proba(X)[:,1]
    q=QUALITY_PROBS_CACHE[stem]
    ranked=[]
    for row,quality in zip(rows,q):
        if row['source']=='v8' and not params['include_v8']:continue
        if row['edge'] and params['guard_edge']:continue
        if row['area']<params['min_area']:continue
        if quality<params['q_cut']:continue
        # Affinity with independent models rewards consistent detections.
        score=float(quality + params['agree_weight']*row['features'][14])
        ranked.append({**row,'rank':score})
    return keep_disjoint(ranked,min_area=params['min_area'],nms_iou=params['nms'],
                         containment=.85,min_free=.55)

from itertools import product
NEW_CONFIGS=[{'q_cut':float(t),'min_area':int(a),'nms':float(d),'agree_weight':float(w),
              'include_v8':bool(include),'guard_edge':bool(guard)}
             for t,a,d,w,include,guard in product((.35,.50,.65), (250,350), (.50,), (.10,),
                                                   ([False,True] if ENABLE_V8 else [False]),(False,True))]
SELECTION_FILE=OUT/'selection.json'
if MODE=='SUBMIT':
    path=SELECTION_FILE if SELECTION_FILE.exists() else PREV10/'selection.json'
    SELECTED=json.loads(path.read_text())
else:
    print('Evaluating baseline controls on',len(SELECT_STEMS),'held-out tune observations')
    orig=score_strategy(SELECT_STEMS,lambda s:old_v4(s,TRAIN))
    v7s=score_strategy(SELECT_STEMS,lambda s:control_masks(s,TRAIN))
    controls={'v4':orig,'v7':v7s}
    reference=('v7' if V7_CONTROL['label']=='saved_v7_exact' and isinstance(V7_CONTROL['strategy'],dict) else max(controls,key=lambda k:controls[k]['pq']))
    refscore=controls[reference]['pq']
    print('Fixed V4 PQ:',round(orig['pq'],6),'restored/reconstructed V7 PQ:',round(v7s['pq'],6))
    for s in tqdm(SELECT_STEMS,desc='Candidate masks on quality-selection tune images'):
        features(s,TRAIN)
    results=[]
    for idx,p in enumerate(tqdm(NEW_CONFIGS,desc='Validate quality-calibrated instance selection')):
        z=score_strategy(SELECT_STEMS,lambda s:rescore(s,TRAIN,p))
        results.append({'index':idx,**p,**z})
    results.append({'index':-1,'kind':'control_v4',**orig})
    results.append({'index':-2,'kind':'control_v7',**v7s})
    df=pd.DataFrame(results).sort_values(['pq','rq'],ascending=False)
    df.to_csv(OUT/'selection_results.csv',index=False)
    display(df.head(15))
    winner=df.iloc[0]
    gain=float(winner['pq']-refscore)
    if int(winner['index'])<0 or gain<MIN_GAIN_TO_PROMOTE:
        strategy={'method':reference}
        print('No validated promotion. Retaining baseline',reference)
    else:
        strategy={'method':'quality','params':NEW_CONFIGS[int(winner['index'])]}
        print('Quality-selector promoted with validation gain:',gain)
    SELECTED={'strategy':strategy,'comparison_baseline':reference,'comparison_pq':refscore,
              'best_selection_pq':float(winner['pq']),'gain':gain,'threshold':MIN_GAIN_TO_PROMOTE,
              'V7_identity':V7_CONTROL['label'],'contract':RUN_CFG,'quality_sha':digest(QMODEL),
              'training_stems':FIT_STEMS,'selection_stems':SELECT_STEMS}
    SELECTION_FILE.write_text(json.dumps(SELECTED,indent=2))

assert SELECTED['contract']==RUN_CFG
if SELECTED['strategy']['method']=='quality':
    print('Selected quality recipe:',SELECTED['strategy']['params'])
else:print('Selected verified fallback:',SELECTED['strategy']['method'])

def infer_selected(stem,images):
    which=SELECTED['strategy']['method']
    if which=='v4':return old_v4(stem,images)
    if which=='v7':return control_masks(stem,images)
    return rescore(stem,images,SELECTED['strategy']['params'])

## 7 — Historical audit (diagnostic only) and strict test RLE submission

The old audit has already been seen in previous runs, so this comparison **does not justify retuning**. Final output contains no overlapping masks, zero-area masks, duplicate IDs or dummy fallback regions. It is computed from the user-owned trained models and actual test JPEGs. We do not overwrite the user's past best submission file.

In [ ]:
if MODE in ('ALL','FIT'):
    audit=sorted(SPLIT['audit'])
    control_a=score_strategy(audit,lambda s:control_masks(s,TRAIN))
    new_a=score_strategy(audit,lambda s:infer_selected(s,TRAIN))
    report={'status':'PREVIOUSLY_SEEN_DIAGNOSTIC_NOT_INDEPENDENT','restored_control':control_a,
            'selected':new_a,'delta_pq':new_a['pq']-control_a['pq'],
            'quality_model_fit_on':len(FIT_STEMS),'selection_on':len(SELECT_STEMS),
            'v7_identity':V7_CONTROL['label']}
    (OUT/'historical_audit.json').write_text(json.dumps(report,indent=2))
    print('Historical audit:',json.dumps(report,indent=2))

if MODE in ('ALL','SUBMIT'):
    rows=[];counts=[]
    for stem in tqdm(sorted(TEST),desc='CPU live instance-quality inference on 180 test frames'):
        rles=infer_selected(stem,TEST)
        assert disjoint(rles),('overlap',stem)
        for j,r in enumerate(rles,1):
            assert int(coco.area(r))>0
            rows.append((f'{stem}_{j}',to_ascii(r)))
        counts.append({'stem':stem,'instances':len(rles)})
    outpath=Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else OUT/'submission.csv'
    table=pd.DataFrame(rows,columns=['filament_id','segmentation_rle'])
    assert not table.empty and table.filament_id.is_unique and not table.isna().any().any()
    table.to_csv(outpath,index=False)
    test=pd.read_csv(outpath,keep_default_na=False)
    assert list(test.columns)==['filament_id','segmentation_rle']
    for stem,group in test.groupby(test.filament_id.str.rsplit('_',n=1).str[0]):
        assert stem in TEST
        rles=[to_rle(t) for t in group.segmentation_rle]
        assert disjoint(rles) and all(int(coco.area(r))>0 for r in rles)
    counts_df=pd.DataFrame(counts)
    counts_df.to_csv(OUT/'test_instances_per_image.csv',index=False)
    end={'rows':len(table),'images':len(TEST),'images_with_predictions':int((counts_df.instances>0).sum()),
         'method':SELECTED['strategy'],'v7_identity':V7_CONTROL['label'],
         'quality_model':str(QMODEL if QMODEL.exists() else PREV10/'mask_quality_model.pkl'),
         'baseline_sha':V4_SHA,'v8_sha':V8_SHA,'csv_sha256':digest(outpath),
         'no_test_labels':True,'no_opaque_predictions':True,'source':'CPU_LIVE_IMAGE_INFERENCE'}
    (OUT/'submission_report.json').write_text(json.dumps(end,indent=2))
    print('FINAL CSV:',outpath,'\n',json.dumps(end,indent=2))
else:
    print('MODE=FIT: model and quality selection saved. Save V10 output and rerun MODE=SUBMIT.')

## 8 — What counts as actual progress

Check `selection_results.csv`, `historical_audit.json`, and `submission_report.json` before submission. Compare the complete actual Kaggle public score against your saved **0.34 V7 submission**, not against V9's weaker 0.31 baseline. A local improvement, even with restored V7 and a genuinely different model, is not a guarantee of public improvement.

**If it fails:** save the generated feature table and false-positive examples rather than running another nearly identical threshold sweep. The next fundamental experiment should be multi-fold out-of-fold instance-ness training (or quality-supervised detector fine-tuning) with more training observations. Label disagreement and different distributions across GONG eras/sites remain limiting factors.

**Acknowledgements:** Mask Scoring R-CNN (CVPR 2019), official Solar Filament Challenge and organizer self-evaluation. This adaptation is not presented as an experimental reproduction of Mask Scoring R-CNN.